In [1]:
import os
os.path.abspath("")

'/home/laser/git/analysisgnn/notebooks'

In [2]:
from pathlib import Path

import pandas as pd
import flexohr # pip install flexohr

try:
    THIS_FOLDER = Path(__file__).parent
except NameError:
    THIS_FOLDER = Path.cwd()
MOZART_PATH = THIS_FOLDER / ".." / "outputs" / "Minuet_in_G_Major_K.1" / "reference_mean.csv"

In [3]:
mozart = pd.read_csv(MOZART_PATH, index_col=0)
mozart

,note_id,onset_beat,measure,duration_beat,pitch_spelling,pitch_midi,cadence,cadence_confidence,localkey,localkey_confidence,...,tpc_is_root,tpc_is_root_confidence,tpc_is_bass,tpc_is_bass_confidence,downbeat,downbeat_confidence,note_degree,note_degree_confidence,staff,staff_confidence
row,,,,,,,,,,,,,,,,,,,,,
0,p0n0,-1.0,1,0.5,B4,83,NaN,0.926015,G,0.936767,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
1,p0n2,-0.5,1,0.5,G4,79,NaN,0.926015,G,0.936767,...,True,0.901820,True,0.640260,0,0.893046,1,0.946128,0,0.385278
2,p0n4,0.0,2,1.0,G2,55,NaN,0.926015,G,0.936767,...,True,0.897312,True,0.701393,2,0.313901,1,0.937878,3,0.324365
3,p0n3,0.0,2,1.0,B3,71,NaN,0.926015,G,0.936767,...,False,0.957455,False,0.758374,2,0.326903,3,0.941654,3,0.382109
4,p0n6,1.0,2,1.0,A2,57,NaN,0.926015,G,0.936767,...,False,0.636777,True,0.567192,2,0.363544,2,0.645440,3,0.548150
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,p0n259,92.0,35,1.0,G1,43,NaN,0.709249,C,0.924959,...,True,0.952441,True,0.939823,3,0.385908,5,0.948357,2,0.316436
252,p0n258,92.0,35,1.0,B3,71,NaN,0.709249,C,0.924959,...,False,0.964105,False,0.951975,3,0.379571,7,0.942371,2,0.364505
253,p0n261,93.0,36,2.0,C2,48,PAC,0.756502,C,0.911427,...,True,0.953619,True,0.940924,1,0.859988,1,0.937045,2,0.497355


In [4]:
def unpivot_with_builtin(df, tasks):
    # 1. Rename columns to have a consistent "metric_task" format
    rename_mapping = {}
    for task in tasks:
        rename_mapping[task] = f"value_{task}"
        rename_mapping[f"{task}_confidence"] = f"confidence_{task}"

    df_renamed = df.rename(columns=rename_mapping)

    # 2. Identify the ID variables (the columns that aren't tasks/confidences)
    id_vars = [col for col in df_renamed.columns if not col.startswith(('value_', 'confidence_'))]

    # 3. Apply the built-in wide_to_long
    result_df = pd.wide_to_long(
        df_renamed,
        stubnames=['value', 'confidence'], # The new column names you want
        i=id_vars,                         # The columns to keep and duplicate
        j='task',                          # The name of the new ID column
        sep='_',                           # The separator between metric and task
        suffix=r'\w+'                      # Regex to match the task names
    )
    result_df = result_df.reset_index()
    id_vars.remove("note_id")
    return pd.concat([result_df[["note_id", "task", "value", "confidence"]],result_df[id_vars]], axis=1)

long_mozart = unpivot_with_builtin(mozart, tasks=["localkey", "quality", "inversion", "degree1", "degree2", "root", "bass", "tonkey"])
long_mozart

,note_id,task,value,confidence,onset_beat,measure,duration_beat,pitch_spelling,pitch_midi,cadence,...,tpc_is_root,tpc_is_root_confidence,tpc_is_bass,tpc_is_bass_confidence,downbeat,downbeat_confidence,note_degree,note_degree_confidence,staff,staff_confidence
0,p0n0,localkey,G,0.936767,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
1,p0n0,tonkey,G,0.905978,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
2,p0n0,quality,major triad,0.888394,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
3,p0n0,inversion,0,0.746613,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
4,p0n0,root,G,0.860512,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2043,p0n262,inversion,0,0.927335,94.0,36,1.0,C4,72,NaN,...,True,0.954334,True,0.933223,2,0.490026,1,0.945832,2,0.394236
2044,p0n262,root,C,0.893067,94.0,36,1.0,C4,72,NaN,...,True,0.954334,True,0.933223,2,0.490026,1,0.945832,2,0.394236
2045,p0n262,bass,C,0.910163,94.0,36,1.0,C4,72,NaN,...,True,0.954334,True,0.933223,2,0.490026,1,0.945832,2,0.394236
2046,p0n262,degree1,1,0.927791,94.0,36,1.0,C4,72,NaN,...,True,0.954334,True,0.933223,2,0.490026,1,0.945832,2,0.394236


In [5]:
for note_id, group in long_mozart.groupby("note_id"):
    display(group.sort_values("confidence", ascending=False))
    break

,note_id,task,value,confidence,onset_beat,measure,duration_beat,pitch_spelling,pitch_midi,cadence,...,tpc_is_root,tpc_is_root_confidence,tpc_is_bass,tpc_is_bass_confidence,downbeat,downbeat_confidence,note_degree,note_degree_confidence,staff,staff_confidence
0,p0n0,localkey,G,0.936767,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
1,p0n0,tonkey,G,0.905978,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
2,p0n0,quality,major triad,0.888394,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
7,p0n0,degree2,NaN,0.872240,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
6,p0n0,degree1,1,0.863107,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
4,p0n0,root,G,0.860512,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
3,p0n0,inversion,0,0.746613,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835
5,p0n0,bass,G,0.604524,-1.0,1,0.5,B4,83,NaN,...,False,0.941229,False,0.731558,1,0.621634,3,0.926605,3,0.317835


In [6]:
tonicized_mask = mozart.tonkey != mozart.localkey
degree2_mask = mozart.degree2.notna()
mozart[tonicized_mask & ~degree2_mask]

,note_id,onset_beat,measure,duration_beat,pitch_spelling,pitch_midi,cadence,cadence_confidence,localkey,localkey_confidence,...,tpc_is_root,tpc_is_root_confidence,tpc_is_bass,tpc_is_bass_confidence,downbeat,downbeat_confidence,note_degree,note_degree_confidence,staff,staff_confidence
row,,,,,,,,,,,,,,,,,,,,,
180,p0n187,71.5,28,0.5,G2,55,NaN,0.709249,C,0.628031,...,False,0.904725,False,0.827497,0,0.887024,5,0.488772,2,0.353358
181,p0n186,71.5,28,0.5,E4,76,NaN,0.709249,C,0.628031,...,True,0.559189,True,0.645594,0,0.886762,3,0.321173,2,0.337943
